# Target-gene coefficient similarity

**Question:** do two target genes get *modulated the same way* by the same factors, cell-by-cell?

Each trained gene `G` has a per-cell coefficient map for every modulator, stored in
`adata.obsm['beta_<G>']` (`cells x modulators`) with names in
`adata.uns['beta_modulators'][G]`. Modulator group is read from the name separator
(`@`=metab, `$`=lr, `#`=ltf, none=tf) — same convention as `beta_analysis.py`.

### The one primitive: per-factor similarity
For a pair of genes and one modulator, take each gene's per-cell coefficient vector
(absent modulator / unfit cell = 0). With `standardize=True` we z-score each vector on
its own (never pooling scales across factors) and the value is the **Pearson r ∈ [-1, 1]**
of the two genes' spatial coefficient maps for that factor. `standardize=False` gives the
raw dot product.

### Aggregate to a single number
`similarity()` collapses the per-factor vector to one score:
- standardized → **mean r over factors that are active (non-zero) in both genes**, so the
  score stays on the same [-1, 1] scale no matter how many zeroed factors there are.
- raw → sum of the per-factor dots (Frobenius block dot).

> Data source: `spacetravlr_adata.h5ad` (built once by `beta_analysis.betas_to_adata`),
> which lives on Savio — the real cells below run there. The synthetic cell needs only
> numpy/pandas and runs anywhere.

In [ ]:
import sys
from pathlib import Path
# Make the repo root importable regardless of CWD or machine.
_start = Path.cwd()
_root = next((p for p in (_start, *_start.parents)
              if (p / ".git").exists() or (p / "setup.py").exists()), _start)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## The functions

In [ ]:
# Modulator group from the name separator (matches beta_analysis.SEPARATORS).
SEPARATORS = {"@": "metab", "$": "lr", "#": "ltf"}


def _group(modulator):
    """'metab', 'lr', 'ltf', or 'tf' (no separator)."""
    for sep, name in SEPARATORS.items():
        if sep in modulator:
            return name
    return "tf"


def gene_block(adata, gene, group="metab"):
    """One gene's coefficients as a (cells x modulators) DataFrame, restricted to `group`.

    `group` is one of 'metab', 'lr', 'ltf', 'tf', or None (=all). Unfit cells are NaN in
    obsm; we fill them (and any absent modulator) with 0 — 'not included' means 0.
    """
    names = list(adata.uns["beta_modulators"].get(gene, []))
    keep = [i for i, n in enumerate(names) if group is None or _group(n) == group]
    block = pd.DataFrame(
        np.asarray(adata.obsm[f"beta_{gene}"])[:, keep],
        index=pd.Index(adata.obs_names),
        columns=[names[i] for i in keep],
    )
    return block.fillna(0.0)


def _zscore(v):
    """z-score a 1-D array; a constant / all-zero vector -> zeros (factor contributes nothing)."""
    sd = v.std()  # numpy std => population (ddof=0)
    if sd == 0:
        return np.zeros_like(v)
    return (v - v.mean()) / sd

In [ ]:
def factor_sims(adata, gene_a, gene_b, group="metab", standardize=True):
    """Per-modulator similarity between two genes' coefficient maps (the reusable primitive).

    Returns a Series indexed by modulator. Each value is the Pearson r in [-1, 1]
    (standardize=True) or the raw dot product (False) of the two genes' per-cell
    coefficients for that modulator. Each factor is z-scored on its own, so scales are
    never pooled across factors/groups. A factor absent (or constant) in either gene is 0.
    """
    A = gene_block(adata, gene_a, group)
    B = gene_block(adata, gene_b, group)
    mods = A.columns.union(B.columns)
    A = A.reindex(columns=mods, fill_value=0.0)
    B = B.reindex(columns=mods, fill_value=0.0)

    out = {}
    for m in mods:
        a, b = A[m].to_numpy(), B[m].to_numpy()
        if standardize:
            a, b = _zscore(a), _zscore(b)
            out[m] = float(a @ b) / len(a)  # Pearson r (0 if either factor is constant)
        else:
            out[m] = float(a @ b)
    return pd.Series(out, name=f"{gene_a}~{gene_b}")


def similarity(adata, gene_a, gene_b, group="metab", standardize=True):
    """Collapse the per-factor similarities to one gene-gene score.

    standardize=True: mean Pearson r over factors active (non-zero) in both genes, so the
    score is on the same [-1, 1] scale regardless of how many factors were zeroed out.
    Returns 0.0 if no factor is shared. standardize=False: sum of the raw per-factor dots.
    """
    v = factor_sims(adata, gene_a, gene_b, group, standardize)
    if not standardize:
        return float(v.sum())
    active = v[v != 0.0]  # a factor is 0 exactly when it is absent/constant in a gene
    return float(active.mean()) if len(active) else 0.0

In [ ]:
def similarity_grid(adata, genes, group="metab", standardize=True):
    """Symmetric genes x genes similarity DataFrame (each entry via `similarity`)."""
    genes = list(genes)
    M = np.zeros((len(genes), len(genes)))
    for i in range(len(genes)):
        for j in range(i, len(genes)):
            M[i, j] = M[j, i] = similarity(adata, genes[i], genes[j], group, standardize)
    return pd.DataFrame(M, index=genes, columns=genes)


def _heatmap(ax, mat, center0=True, annot=True):
    """Bare-bones labelled heatmap (matplotlib only). Diverging + centered at 0 for r."""
    data = mat.to_numpy(dtype=float)
    if center0:
        lim = np.nanmax(np.abs(data)) or 1.0
        im = ax.imshow(data, cmap="RdBu_r", vmin=-lim, vmax=lim)
    else:
        im = ax.imshow(data, cmap="viridis")
    ax.set_xticks(range(mat.shape[1]), mat.columns, rotation=90)
    ax.set_yticks(range(mat.shape[0]), mat.index)
    if annot:
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                if np.isfinite(data[i, j]):
                    ax.text(j, i, f"{data[i, j]:.2f}", ha="center", va="center", fontsize=7)
    return im


def plot_grid(mat, title="", triangular=True, center0=True):
    """Heatmap of a genes x genes similarity matrix; `triangular` masks the upper half (diag view)."""
    mat = mat.copy()
    if triangular:
        mask = np.triu(np.ones(mat.shape, bool), k=1)
        mat = mat.mask(mask)
    fig, ax = plt.subplots(figsize=(0.7 * mat.shape[1] + 2, 0.7 * mat.shape[0] + 2))
    im = _heatmap(ax, mat, center0=center0)
    fig.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(title)
    fig.tight_layout()
    return fig


def plot_factor_sims(series_or_df, title="", center0=True):
    """Vector heatmap of per-factor similarities.

    Pass one `factor_sims(...)` Series, or a DataFrame (modulators x gene-pairs) built by
    concatenating several — to see *which* factors drive a correlation.
    """
    df = series_or_df.to_frame() if isinstance(series_or_df, pd.Series) else series_or_df
    fig, ax = plt.subplots(figsize=(0.7 * df.shape[1] + 2, 0.3 * df.shape[0] + 2))
    im = _heatmap(ax, df, center0=center0)
    fig.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(title)
    fig.tight_layout()
    return fig

## Synthetic sanity check (runs anywhere — numpy/pandas only)

A 4-cell, 2-gene mock with hand-computed answers, so we trust the math before touching
the real data. `B`'s `GLN@SLC` map is exactly `2 x A`'s (perfectly correlated → r=1);
`A` also has a `LAC@MCT` map that `B` lacks (inactive → contributes 0).

In [ ]:
from types import SimpleNamespace

mock = SimpleNamespace(
    obs_names=["c0", "c1", "c2", "c3"],
    obsm={
        # columns follow uns['beta_modulators'] order below
        "beta_GENE_A": np.array([[1.0, 1.0], [2, 0], [3, 1], [4, 0]]),  # GLN@SLC, LAC@MCT
        "beta_GENE_B": np.array([[2.0], [4], [6], [8]]),               # GLN@SLC only (=2x A)
    },
    uns={"beta_modulators": {"GENE_A": ["GLN@SLC", "LAC@MCT"], "GENE_B": ["GLN@SLC"]}},
)

v = factor_sims(mock, "GENE_A", "GENE_B", group="metab", standardize=True)
print("per-factor r:\n", v)
assert np.isclose(v["GLN@SLC"], 1.0), v["GLN@SLC"]   # perfectly correlated
assert np.isclose(v["LAC@MCT"], 0.0), v["LAC@MCT"]   # absent in B -> inactive

assert np.isclose(similarity(mock, "GENE_A", "GENE_B", standardize=True), 1.0)  # mean over active = 1
# raw: dot([1,2,3,4],[2,4,6,8]) = 60; LAC@MCT dots with zeros = 0
assert np.isclose(similarity(mock, "GENE_A", "GENE_B", standardize=False), 60.0)
print("\nOK — synthetic checks pass")

## Apply to the melanoma dataset (Savio)

Loads the frozen `spacetravlr_adata.h5ad` and runs the grid over `FOCUS_GENES`.

In [ ]:
import scanpy as sc
from metab_processing.metab_travlr_config import PROJECT_DATA_DIR, FOCUS_GENES

DATASET_NAME = "Primary_Dermal_Melanoma"
DATA_DIR = f"{PROJECT_DATA_DIR}/{DATASET_NAME}"

adata = sc.read_h5ad(f"{DATA_DIR}/spacetravlr_adata.h5ad")
adata

In [ ]:
# Which of our focus genes were actually trained (have a betas block)?
genes = [g for g in FOCUS_GENES if g in adata.uns["beta_modulators"]]
print(f"{len(genes)}/{len(FOCUS_GENES)} focus genes present:", genes)

In [ ]:
# Standardized (Pearson-r) grid on the metabolite factors.
grid = similarity_grid(adata, genes, group="metab", standardize=True)
plot_grid(grid, title=f"{DATASET_NAME}: metab-coefficient similarity (mean r)");

In [ ]:
# Drill into one pair: which metabolites drive (or oppose) the correlation?
GENE_A, GENE_B = genes[0], genes[1]
v = factor_sims(adata, GENE_A, GENE_B, group="metab", standardize=True)
plot_factor_sims(v[v != 0].sort_values(), title=f"{GENE_A} ~ {GENE_B}: per-metabolite r");